1 — Chargement et vérification qualité

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Chargement
df_renouv = pd.read_csv("data/renouvelables.csv")
df_elec = pd.read_csv("data/green_bonds.csv")

print("=== QUALITÉ DATASET 1 — Renouvelables ===")
print(f"Lignes : {len(df_renouv)}")
print(f"Doublons : {df_renouv.duplicated().sum()}")
print(f"Valeurs manquantes :\n{df_renouv.isnull().sum()}")

print("\n=== QUALITÉ DATASET 2 — Électricité ===")
print(f"Lignes : {len(df_elec)}")
print(f"Doublons : {df_elec.duplicated().sum()}")
print(f"Valeurs manquantes :\n{df_elec.isnull().sum()}")

=== QUALITÉ DATASET 1 — Renouvelables ===
Lignes : 150
Doublons : 0
Valeurs manquantes :
economy    0
year       0
value      0
country    0
dtype: int64

=== QUALITÉ DATASET 2 — Électricité ===
Lignes : 120
Doublons : 0
Valeurs manquantes :
economy              0
year                 0
elec_renouvelable    0
country              0
dtype: int64


✅ Données très propres !
0 doublons · 0 valeurs manquantes sur les deux datasets 

2 — Standardisation et enrichissement

In [2]:
# Renommer les colonnes pour plus de clarté
df_renouv = df_renouv.rename(columns={"value": "renouvelables_pct"})
df_elec = df_elec.rename(columns={"elec_renouvelable": "elec_renouvelable_pct"})

# Ajouter une colonne région
regions = {
    "Sweden": "Nordic", "Denmark": "Nordic", "Austria": "Central",
    "Germany": "Central", "Netherlands": "Western", "Belgium": "Western",
    "France": "Western", "Spain": "Southern", "Italy": "Southern",
    "Poland": "Eastern"
}
df_renouv["region"] = df_renouv["country"].map(regions)
df_elec["region"] = df_elec["country"].map(regions)

# Ajouter objectif UE 2030
df_renouv["objectif_ue_2030"] = 42.5
df_elec["objectif_ue_2030"] = 42.5

# Écart par rapport à l'objectif
df_renouv["ecart_objectif"] = df_renouv["renouvelables_pct"] - df_renouv["objectif_ue_2030"]
df_elec["ecart_objectif"] = df_elec["elec_renouvelable_pct"] - df_elec["objectif_ue_2030"]

print("=== DONNÉES ENRICHIES ===")
display(df_renouv.head(5))
print(f"\nNouvelles colonnes : {list(df_renouv.columns)}")

=== DONNÉES ENRICHIES ===


,economy,year,renouvelables_pct,country,region,objectif_ue_2030,ecart_objectif
0,AUT,2007,29.0,Austria,Central,42.5,-13.5
1,BEL,2007,3.4,Belgium,Western,42.5,-39.1
2,DEU,2007,10.5,Germany,Central,42.5,-32.0
3,DNK,2007,17.7,Denmark,Nordic,42.5,-24.8
4,ESP,2007,9.0,Spain,Southern,42.5,-33.5



Nouvelles colonnes : ['economy', 'year', 'renouvelables_pct', 'country', 'region', 'objectif_ue_2030', 'ecart_objectif']


✅ On est passé de 4 à 7 colonnes !
Ce qu'on a ajouté :

region → Nordic, Central, Western, Southern, Eastern
objectif_ue_2030 → 42.5% pour tous
ecart_objectif → distance par rapport à l'objectif UE

3 — Sauvegarde des données nettoyées

In [3]:
# Sauvegarde
df_renouv.to_csv("data/renouvelables_clean.csv", index=False)
df_elec.to_csv("data/electricite_clean.csv", index=False)

print("✅ Données nettoyées sauvegardées !")
print(f"\nRécapitulatif du nettoyage :")
print(f"  - Doublons supprimés     : 0 (données déjà propres)")
print(f"  - Valeurs manquantes     : 0")
print(f"  - Nouvelles colonnes     : region, objectif_ue_2030, ecart_objectif")
print(f"  - Dataset 1 final        : {len(df_renouv)} lignes · {len(df_renouv.columns)} colonnes")
print(f"  - Dataset 2 final        : {len(df_elec)} lignes · {len(df_elec.columns)} colonnes")

# Pays en retard sur l'objectif UE 2030 (dernière année disponible)
derniere_annee = df_renouv[df_renouv["year"] == 2021]
en_retard = derniere_annee[derniere_annee["ecart_objectif"] < 0][["country","renouvelables_pct","ecart_objectif"]]
print(f"\nPays en retard sur l'objectif UE 2030 (42.5%) en 2021 :")
print(en_retard.sort_values("ecart_objectif"))

✅ Données nettoyées sauvegardées !

Récapitulatif du nettoyage :
  - Doublons supprimés     : 0 (données déjà propres)
  - Valeurs manquantes     : 0
  - Nouvelles colonnes     : region, objectif_ue_2030, ecart_objectif
  - Dataset 1 final        : 150 lignes · 7 colonnes
  - Dataset 2 final        : 120 lignes · 7 colonnes

Pays en retard sur l'objectif UE 2030 (42.5%) en 2021 :
         country  renouvelables_pct  ecart_objectif
141      Belgium               11.7           -30.8
147  Netherlands               12.2           -30.3
148       Poland               15.2           -27.3
145       France               16.2           -26.3
146        Italy               17.5           -25.0
142      Germany               17.6           -24.9
144        Spain               19.0           -23.5
140      Austria               36.0            -6.5
143      Denmark               39.5            -3.0


Insight clé :

Belgium et Netherlands les plus en retard (-30%)
Sweden est le seul pays à avoir atteint l'objectif (57.9% > 42.5%)
Denmark et Austria sont les plus proches